&nbsp;

## [ENCODE-rE2G](https://github.com/EngreitzLab/ENCODE_rE2G) Result Parse Notebook (Control-related datasets)

- [An encyclopedia of human enhancer–gene regulatory interactions.](https://doi.org/10.1038/s41586-026-10781-4) で計算されている ENCODE のデータで予測されている enhancer - promoter interaction をパースする notebook
- 熱ストレスに関連していないデータを「Ctrl」条件のデータとして扱っている

In [1]:
import datetime
import os
import pathlib

import polars as pl
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
os.environ["POLARS_MAX_THREADS"] = "16"

In [2]:
execute_date = datetime.datetime.now().strftime("%y%m%d")  # noqa: DTZ005
print(f"Execute Date: {execute_date}")

Execute Date: 261006


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. Parse Target Bed file (Information included: "G"; Gene)

- Target となる領域は [RefSeqCurated.170308.bed.CollapsedGeneBounds.hg38.TSS500bp.bed](https://github.com/EngreitzLab/ENCODE_rE2G/blob/main/reference/RefSeqCurated.170308.bed.CollapsedGeneBounds.hg38.TSS500bp.bed) で定義されている (Download date: 2026-07-31)
- この領域 (TSS 500bp)を通常では遺伝子レベルで表現されているが、これを fanta.bio のBEDファイルとintersect しておくことで、後々の解析をしやすくする
- この領域 (TSS 500bp) と fanta.bio で定義された CRE の重なり具合を算出し、この領域 (TSS 500bp) 側から見たときの割合も算出する (フィルタリング条件に活用するため)
- `ENCODE-rE2G` 側では、範囲が 500 bp と決まっており、この範囲を起点に相互作用が予測されているため、fanta.bio のCREがどの程度カバーしているかを計算しておく
- 同一 `TargetGene` に複数 `cre_id` が紐づく場合は、`tss_cre_overlap_frac_of_tss` が最大の CRE を残す（`pl.when` で明示）
- 同点時は `fantabio_cre_class == "PLA"` を優先し、それでも残る同点は `cre_id` で決定的に1行にする
- 実行コマンド: `cwltool --outdir ./out/ENCODE_rE2G/bedtools_intersect/ ./Tools/bedtools_intersect_v2.cwl ./config/ENCODE_rE2G/encodere2g_refseq_promoter_intersect_fantabio_cre.yml`

In [3]:
# Download https://github.com/EngreitzLab/ENCODE_rE2G/blob/main/reference/RefSeqCurated.170308.bed.CollapsedGeneBounds.hg38.TSS500bp.bed
encodere2g_tatget_refseq_intersect_fantabio_cre = (
    pl.read_csv(
        "../../out/ENCODE_rE2G/bedtools_intersect/RefSeqCurated.170308.bed.CollapsedGeneBounds.hg38.TSS500bp_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed",
        separator="\t",
        has_header=False,
        new_columns=[
            "tss_chr", "tss_start", "tss_end", "TargetGene", "tss_score", "tss_strand",
            "cre_chr", "cre_start", "cre_end", "cre_id", "cre_score", "cre_strand",
            "cre_thick_start", "cre_thick_end", "cre_rgb", "cre_name", "pooled_cre_scores",
            "overlap_bp",  # last column from bedtools -wo
        ]
    ).with_columns(
        pl.col("pooled_cre_scores").str.extract(r"directionality:([^|]+)", 1).cast(pl.Float64, strict=False).alias("directionality"),
        pl.col("pooled_cre_scores").str.extract(r"class:([^|]+)", 1).alias("fantabio_cre_class"),
        pl.col("overlap_bp").alias("tss_cre_overlap_bp"),
    ).with_columns(
        (pl.col("tss_cre_overlap_bp") / (pl.col("tss_end") - pl.col("tss_start")))
        .alias("tss_cre_overlap_frac_of_tss"),
    ).with_columns(
        pl.lit("tss_overlap").alias("target_cre_evidence"),
    ).drop("overlap_bp")
)

pl.Config.set_tbl_rows(2)
display(encodere2g_tatget_refseq_intersect_fantabio_cre)

tss_chr,tss_start,tss_end,TargetGene,tss_score,tss_strand,cre_chr,cre_start,cre_end,cre_id,cre_score,cre_strand,cre_thick_start,cre_thick_end,cre_rgb,cre_name,pooled_cre_scores,directionality,fantabio_cre_class,tss_cre_overlap_bp,tss_cre_overlap_frac_of_tss,target_cre_evidence
str,i64,i64,str,i64,str,str,i64,i64,str,i64,str,i64,i64,str,str,str,f64,str,i64,f64,str
"""chr1""",29120,29620,"""WASH7P""",0,"""-""","""chr1""",29398,29439,"""FCHS_2""",0,""".""",29398,29432,"""255,202,74""","""cp1@WASH7P""","""directionality:1|class:ELA""",1.0,"""ELA""",41,0.082,"""tss_overlap"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",20575460,20575960,"""EIF1AY""",0,"""+""","""chrY""",20575327,20575912,"""FCHS_447281""",0,""".""",20575492,20575772,"""204,0,0""","""cp3@EIF1AY,cp1@TTTY10""","""directionality:0.976236|class:PLA""",0.976236,"""PLA""",452,0.904,"""tss_overlap"""


&nbsp;

&nbsp;

## 1-1: Filter (Information: "G")

- 同一 `TargetGene` に複数 `cre_id` が紐づく場合は、`tss_cre_overlap_frac_of_tss` が最大の CRE を残す（`pl.when` で明示）
- overlap が同点のときは `fantabio_cre_class == "PLA"` を優先する（`pl.when` で `pla_rank` を付与）
- それでも同点のときは `cre_id` の辞書順で決定的に 1 行にする（`sort` + `unique(keep="first")`）
- 分母は ENCODE-rE2G の TSS 500 bp（`tss_end - tss_start`）。閾値による除外は行わず、遺伝子ごとに代表 CRE を 1 本選ぶ

In [4]:
# Keep the CRE with the highest TSS overlap fraction per TargetGene
encodere2g_tatget_refseq_best_cre = (
    encodere2g_tatget_refseq_intersect_fantabio_cre
    .with_columns(
        # Max overlap fraction within each TargetGene
        pl.col("tss_cre_overlap_frac_of_tss")
        .max()
        .over("TargetGene")
        .alias("max_tss_cre_overlap_frac_of_tss"),
    )
    .with_columns(
        # True only for the row(s) that achieve that max
        pl.when(
            pl.col("tss_cre_overlap_frac_of_tss") == pl.col("max_tss_cre_overlap_frac_of_tss")
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("is_best_cre_for_target_gene"),
    )
    .filter(pl.col("is_best_cre_for_target_gene")) # flag

    .with_columns(
        # On ties, prefer PLA (0) over other classes (1)
        pl.when(pl.col("fantabio_cre_class") == "PLA")
        .then(pl.lit(0))
        .otherwise(pl.lit(1))
        .alias("pla_rank"),
    )
    # Tie-break: PLA first, then cre_id for a deterministic single row
    .sort("TargetGene", "pla_rank", "cre_id")
    .unique(subset=["TargetGene"], keep="first")
    .drop("max_tss_cre_overlap_frac_of_tss", "is_best_cre_for_target_gene", "pla_rank")
)

print(
    f"Rows before: {encodere2g_tatget_refseq_intersect_fantabio_cre.height} \n"
    f"Rows after: {encodere2g_tatget_refseq_best_cre.height} \n"
    f"UniqueGenes: {encodere2g_tatget_refseq_best_cre.select('TargetGene').n_unique()}"
)

display(encodere2g_tatget_refseq_best_cre)

Rows before: 38074 
Rows after: 20766 
UniqueGenes: 20766


tss_chr,tss_start,tss_end,TargetGene,tss_score,tss_strand,cre_chr,cre_start,cre_end,cre_id,cre_score,cre_strand,cre_thick_start,cre_thick_end,cre_rgb,cre_name,pooled_cre_scores,directionality,fantabio_cre_class,tss_cre_overlap_bp,tss_cre_overlap_frac_of_tss,target_cre_evidence
str,i64,i64,str,i64,str,str,i64,i64,str,i64,str,i64,i64,str,str,str,f64,str,i64,f64,str
"""chr19""",58353249,58353749,"""A1BG""",0,"""-""","""chr19""",58353431,58353819,"""FCHS_214135""",0,""".""",58353468,58353803,"""0,0,204""","""cp1@A1BG""","""directionality:-0.999634|class:PLA""",-0.999634,"""PLA""",318,0.636,"""tss_overlap"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr1""",77682408,77682908,"""ZZZ3""",0,"""-""","""chr1""",77682463,77683068,"""FCHS_17233""",0,""".""",77682658,77682722,"""0,0,204""","""cp3@ZZZ3""","""directionality:-0.993424|class:PLA""",-0.993424,"""PLA""",445,0.89,"""tss_overlap"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. Integrate `fanta.bio` data (Information included: "E"; enhancer)

- [ENCODE-rE2G](https://github.com/EngreitzLab/ENCODE_rE2G) で得られたエンハンサー候補と、`fanta.bio`のデータを `bedtools intersect` した後の結果をデータフレームとして読み込む
- scatter 版: 複数サンプルの enhancer BED を、共通の fanta.bio CRE peaks と一括 intersect する
- 実行コマンド: `cwltool --outdir ./out/ENCODE_rE2G/bedtools_intersect/enhancer_ctrl/ ./Workflow/bedtools_intersect_v2_scatter.cwl ./config/ENCODE_rE2G/ctrl_re2g_enhancer_intersect_fantabio_cre.yml`

- 各行に `re2g_dataset` / `source_bed_file` を付与し、同一 CRE でも由来サンプルを区別できるようにする
- データが多すぎるので、一旦 TSV として出力した後、それを読み込む 
- 行数が非常に多いため、先にカバー率でフィルタリングしておく

In [ ]:
# # Parse bedtools intersect (-wo) outputs for all enhancer datasets
# BED_INTERSECT_ENHANCER_DIR = pathlib.Path("../../out/ENCODE_rE2G/bedtools_intersect/enhancer_ctrl")

# ENHANCER_INTERSECT_COLUMNS = [
#     "re2g_chr",
#     "re2g_start",
#     "re2g_end",
#     "cre_chr",
#     "cre_start",
#     "cre_end",
#     "cre_id",
#     "cre_score",
#     "cre_strand",
#     "cre_thick_start",
#     "cre_thick_end",
#     "cre_rgb",
#     "cre_name",
#     "pooled_cre_scores",
#     "overlap_bp",  # last column from bedtools -wo
# ]


# def read_enhancer_intersect_bed(bed_path: pathlib.Path) -> pl.DataFrame:
#     """
#     Read one enhancer x fanta.bio intersect BED and tag dataset ID from filename.
#     Example stem prefix: ENCSR000MZA_ENCFF654AMU
#     """
#     re2g_dataset = bed_path.stem.split("_only_enhancer", 1)[0]

#     return (
#         pl.read_csv(
#             bed_path,
#             separator="\t",
#             has_header=False,
#             new_columns=ENHANCER_INTERSECT_COLUMNS,
#             n_threads=16,
#         )
#         .with_columns(
#             pl.col("pooled_cre_scores")
#             .str.extract(r"directionality:([^|]+)", 1)
#             .cast(pl.Float64, strict=False)
#             .alias("directionality"),
#             pl.col("pooled_cre_scores")
#             .str.extract(r"class:([^|]+)", 1)
#             .alias("fantabio_cre_class"),
#             pl.col("overlap_bp").alias("enhancer_cre_overlap_bp"),
#         )
#         .with_columns(
#             (pl.col("enhancer_cre_overlap_bp") / (pl.col("cre_end") - pl.col("cre_start"))).alias("enhancer_cre_overlap_frac_of_fantabio_cre"),
#             # Categorical saves memory because the same string repeats on every row
#             pl.lit(re2g_dataset).cast(pl.Categorical).alias("re2g_dataset"),
#             pl.lit(bed_path.name).cast(pl.Categorical).alias("source_bed_file"),
#         )
#         .drop(
#             "cre_score",
#             "cre_strand",
#             "cre_rgb",
#             "cre_thick_start",
#             "cre_thick_end",
#             "pooled_cre_scores",
#             "overlap_bp",
#         )
#     )


# bed_paths = sorted(BED_INTERSECT_ENHANCER_DIR.glob("*_intersecting_*.bed"))

# enhancer_fanta_intersect = pl.concat(
#     [read_enhancer_intersect_bed(p) for p in bed_paths],
#     how="vertical_relaxed",
# ).sort("cre_chr", "cre_start", "re2g_dataset")

# print(f"files: {len(bed_paths)}")
# print(f"datasets: {enhancer_fanta_intersect.select('re2g_dataset').n_unique()}")

# pl.Config.set_tbl_rows(14)
# display(
#     enhancer_fanta_intersect.group_by("re2g_dataset")
#     .len()
#     .sort("re2g_dataset")
# )
# pl.Config.set_tbl_rows(2)
# display(enhancer_fanta_intersect)

In [ ]:
# # データが多すぎるので、ファイルにしておく
# enhancer_fanta_intersect.write_csv(
#     "../../out/ENCODE_rE2G/ctrl_list/encode_re2g_ctrl_enhancer_fanta_intersect.tsv.gz",
#     separator="\t",
#     compression = "gzip",
#     compression_level = 9
# )

In [9]:
# sample metadata
encode_re2g_ctrl_metadata = pl.read_csv(
    "../../out/ENCODE_rE2G/ctrl_list/metadata/ENCODE-rE2G_bedpe_map_supplementary_table_12_join.tsv",
    separator="\t",
    n_threads = 16
).select(
    "ENCODE-rE2G accession",
    "files_accession",
    "Model",
    "Sample",
    "Biosample term name",
    "Biosample type",
    "Biosample term id_table12"
).with_columns(
    (pl.col("ENCODE-rE2G accession") + "_" + pl.col("files_accession")).alias("re2g_dataset")
).drop(
    "ENCODE-rE2G accession",
    "files_accession"
).with_columns(
    # All columns repeat on every row after joining to enhancer_fanta_intersect;
    # re2g_dataset must also be Categorical to match the join key dtype
    # 全部 Categorical にすることでメモリを減らす
    pl.col(pl.String).cast(pl.Categorical)
)

display(encode_re2g_ctrl_metadata)

Model,Sample,Biosample term name,Biosample type,Biosample term id_table12,re2g_dataset
cat,cat,cat,cat,cat,cat
"""Extended""","""GM12878_ENCSR000EMT""","""GM12878""","""cell line""","""EFO:0002784""","""ENCSR997TVR_ENCFF547ARY"""
…,…,…,…,…,…
"""DNase-only""","""vagina_ENCSR437AYW""","""vagina""","""tissue""","""UBERON:0000996""","""ENCSR738AUI_ENCFF167ZIP"""


In [10]:
# About "Categorical", see: https://pola.rs/posts/categoricals-refactor/
enhancer_fanta_intersect = pl.scan_csv(
    "../../out/ENCODE_rE2G/ctrl_list/encode_re2g_ctrl_enhancer_fanta_intersect.tsv.gz",
    separator="\t",
    # Categorical saves memory for low-cardinality strings repeated on every row
    schema_overrides={
        "re2g_chr": pl.Categorical,
        "cre_chr": pl.Categorical,
        "re2g_dataset": pl.Categorical,
    }
).filter(
    # Same condition as section 5; applied here because the section 4 join runs out of memory otherwise
    # 行数が多いため、先に 0.5 以上のデータをフィルタリングしておく
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre") >= 0.5
).drop(
    "source_bed_file"
).collect().join(

    # sample などの metadata を join する
    encode_re2g_ctrl_metadata,
    on = "re2g_dataset",
    how = "left",
    validate = "m:1"
)

display(enhancer_fanta_intersect)

re2g_chr,re2g_start,re2g_end,cre_chr,cre_start,cre_end,cre_id,cre_name,directionality,fantabio_cre_class,enhancer_cre_overlap_bp,enhancer_cre_overlap_frac_of_fantabio_cre,re2g_dataset,Model,Sample,Biosample term name,Biosample type,Biosample term id_table12
cat,i64,i64,cat,i64,i64,str,str,f64,str,i64,f64,cat,cat,cat,cat,cat,cat
"""chr1""",778470,778970,"""chr1""",778536,778928,"""FCHS_19""","""cp5@LINC00115,cp2@LINC01409""",-0.304838,"""PLA""",392,1.0,"""ENCSR000MZA_ENCFF654AMU""","""DNase-only""","""T-cell_ENCSR926SNI""","""T-cell""","""primary cell""","""CL:0000084"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrX""",156028746,156029664,"""chrX""",156029023,156029480,"""FCHS_593847""","""FCHS_593847""",0.79661,"""ELA""",457,1.0,"""ENCSR624FDG_ENCFF200PTU""","""DNase-only""","""placenta_ENCSR503RWO""","""placenta""","""tissue""","""UBERON:0001987"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 3. ENCODE rE2G `bed` file (Information Included: "E" / "G")

- このデータは、予測された enhancer - gene interaction が記載されており、`join` する際の 核となるデータセットとして使用される
- `bedpe` 形式とは別の`bed 3+` 形式のデータを読みこんで解析することで、`fanta.bio`との `bedtools intersect`したデータフレームと、それぞれ`join`するための核となるデータとする
- 入力: ENCODE portal からダウンロードしたファイル: [../../out/ENCODE_rE2G/ctrl_list/ENCODE-rE2G_ctrl_enhancer_gene_interaction/](../../out/ENCODE_rE2G/ctrl_list/ENCODE-rE2G_ctrl_enhancer_gene_interaction/)
- 未フィルタ TSV は score 分布が 0 付近に極端に偏るため、プロジェクト既定の threshold 適用後ファイルを用いる
- セクション2の `enhancer_fanta_intersect` と join できるよう、同じ規則で `re2g_dataset` を付与するが、これは 事前にマッピングしておいたファイルを利用する

In [12]:
# ID mapping file
accession_bed_map = pl.read_csv(
    "../../out/ENCODE_rE2G/ctrl_list/metadata/encode_re2g_bed_bedpe_map.tsv",
    separator="\t",
    n_threads = 16
).with_columns(
    pl.col(pl.String).cast(pl.Categorical) # 全部 Categorical にすることでメモリを減らす
)

display(accession_bed_map)

re2g_dataset,annotation_accession,bed_accession,bedpe_accession,derived_from_match
cat,cat,cat,cat,bool
"""ENCSR000MZA_ENCFF654AMU""","""ENCSR000MZA""","""ENCFF727YIH""","""ENCFF654AMU""",true
…,…,…,…,…
"""ENCSR999YQC_ENCFF625FFH""","""ENCSR999YQC""","""ENCFF553ZXL""","""ENCFF625FFH""",true


In [11]:
# test (このデータを concat して、ENCODE-rE2G のデータセットを作成する)
ENCFF003BKC = pl.read_csv(
    "../../out/ENCODE_rE2G/ctrl_list/ENCODE-rE2G_ctrl_enhancer_gene_interaction/ENCFF003BKC.bed.gz",
    separator="\t",
    n_threads = 16
)

pl.Config.set_tbl_rows(2)
display(ENCFF003BKC)

#chr,start,end,name,class,TargetGene,TargetGeneEnsemblID,TargetGeneTSS,isSelfPromoter,CellType,distanceToTSS.Feature,normalizedDNase_prom.Feature,3DContact.Feature,ABC.Score.Feature,numCandidateEnhGene.Feature,numTSSEnhGene.Feature,sumNearbyEnhancers.Feature,ubiquitousExpressedGene.Feature,Score
str,i64,i64,str,str,str,str,i64,bool,str,f64,f64,f64,f64,i64,i64,f64,bool,f64
"""chr1""",778465,778965,"""chr1:778465-778965""","""intergenic""","""LINC00115""","""ENSG00000225880""",827522,false,"""head of caudate nucleus""",48807.0,0.591285,0.099048,0.199458,2,4,89.573841,false,0.75126
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrX""",155881029,155881533,"""chrX:155881029-155881533""","""promoter""","""VAMP7""","""ENSG00000124333""",155881279,true,"""head of caudate nucleus""",2.0,0.761952,0.259196,1.0,0,0,0.0,false,0.999999


In [13]:
# Parse ENCODE-rE2G thresholded bed3+ files downloaded from the ENCODE portal
RE2G_BED_DIR = pathlib.Path("../../out/ENCODE_rE2G/ctrl_list/ENCODE-rE2G_ctrl_enhancer_gene_interaction")

# Columns shared by all files (2 "Extended" model files have many extra feature columns)
RE2G_BED_COLUMNS = [
    "#chr",
    "start",
    "end",
    "class",
    "TargetGene",
    "TargetGeneEnsemblID",
    "TargetGeneTSS",
    "isSelfPromoter",
    "CellType",
    "distanceToTSS.Feature",
    "normalizedDNase_prom.Feature",
    "3DContact.Feature",
    "numCandidateEnhGene.Feature",
    "numTSSEnhGene.Feature",
    "sumNearbyEnhancers.Feature",
    "Score",
]

# bed3+ accession (e.g. ENCFF003BKC) -> re2g_dataset used in section 2 (e.g. ENCSR297HTV_ENCFF379TOS)
bed_to_re2g_dataset = dict(
    zip(accession_bed_map["bed_accession"], accession_bed_map["re2g_dataset"])
)


def read_re2g_prediction_bed(bed_path: pathlib.Path) -> pl.DataFrame:
    """
    Read one thresholded ENCODE-rE2G bed3+ file and tag re2g_dataset via accession_bed_map.
    Output columns match the downstream join (section 4) and the HS notebook where possible.
    """
    bed_accession = bed_path.name.split(".", 1)[0]
    re2g_dataset = bed_to_re2g_dataset[bed_accession]

    return (
        pl.read_csv(
            bed_path,
            separator="\t",
            columns=RE2G_BED_COLUMNS,
            n_threads=16,
        )
        .select(
            # Categorical join keys must match section 2 (enhancer_fanta_intersect)
            pl.col("#chr").cast(pl.Categorical).alias("re2g_chr"),
            pl.col("start").alias("re2g_start"),
            pl.col("end").alias("re2g_end"),
            pl.col("class").cast(pl.Categorical),
            # Keep String: section 6 joins TargetGene with a String column
            "TargetGene",
            "isSelfPromoter",
            pl.col("distanceToTSS.Feature").alias("distanceToTSS"),
            pl.col("Score").alias("ENCODE-rE2G.Score"),
            pl.struct(
                "TargetGeneTSS",
                pl.col("TargetGeneEnsemblID").alias("TargetGeneEnsembl_ID"),
                pl.col("normalizedDNase_prom.Feature").alias("normalizedDNase_prom"),
                pl.col("3DContact.Feature").alias("3DContact"),
                pl.col("numCandidateEnhGene.Feature").alias("numCandidateEnhGene"),
                pl.col("numTSSEnhGene.Feature").alias("numTSSEnhGene"),
                pl.col("sumNearbyEnhancers.Feature").alias("sumNearbyEnhancers"),
                pl.col("CellType").cast(pl.Categorical),
            ).alias("re2g_features"),
            pl.lit(re2g_dataset).cast(pl.Categorical).alias("re2g_dataset"),
            pl.lit(bed_accession).cast(pl.Categorical).alias("bed_accession"),
        )
        .unique()
    )

In [14]:
# Require exactly one downloaded bed3+ file per entry in accession_bed_map
bed_paths = sorted(RE2G_BED_DIR.glob("ENCFF*.bed.gz"))

bed_accessions = {p.name.split(".", 1)[0] for p in bed_paths}
map_accessions = set(bed_to_re2g_dataset)

if bed_accessions != map_accessions:
    raise RuntimeError(
        f"Not in accession_bed_map: {sorted(bed_accessions - map_accessions)}\n"
        f"Not downloaded: {sorted(map_accessions - bed_accessions)}"
    )

encode_re2g_predictions = pl.concat(
    [read_re2g_prediction_bed(p) for p in bed_paths],
    how="vertical_relaxed",
)
# Sorting 117M rows costs several GB and is not required for the section 4 join

pl.Config.set_tbl_rows(14)
print(f"files: {len(bed_paths)}")
print(f"datasets: {encode_re2g_predictions.select('re2g_dataset').n_unique()}")
display(
    encode_re2g_predictions.group_by("re2g_dataset")
    .len()
    .sort("re2g_dataset")
)

# Optional sanity check against section 2 dataset IDs
if "enhancer_fanta_intersect" in globals():
    bed_ids = set(enhancer_fanta_intersect["re2g_dataset"].unique().to_list())
    tsv_ids = set(encode_re2g_predictions["re2g_dataset"].unique().to_list())
    print(f"dataset ID match with section 2: {bed_ids == tsv_ids}")
    if bed_ids != tsv_ids:
        print("only in BED:", sorted(bed_ids - tsv_ids))
        print("only in TSV:", sorted(tsv_ids - bed_ids))


display(encode_re2g_predictions)

files: 1460
datasets: 1460


re2g_dataset,len
cat,u32
"""ENCSR000MZA_ENCFF654AMU""",85686
"""ENCSR002YWN_ENCFF435DPV""",86425
"""ENCSR004ZDG_ENCFF053LFW""",85074
"""ENCSR006OUR_ENCFF317WUF""",86315
"""ENCSR006VBD_ENCFF418STV""",82774
"""ENCSR007SKZ_ENCFF645YJN""",86522
"""ENCSR010JRN_ENCFF554KGH""",81354
…,…
"""ENCSR997TVR_ENCFF547ARY""",97300


dataset ID match with section 2: True


re2g_chr,re2g_start,re2g_end,class,TargetGene,isSelfPromoter,distanceToTSS,ENCODE-rE2G.Score,re2g_features,re2g_dataset,bed_accession
cat,i64,i64,cat,str,bool,f64,f64,struct[8],cat,cat
"""chr19""",3665780,3666556,"""genic""","""PIP5K1C""",false,34311.0,0.557754,"{3700479,""ENSG00000186111"",0.717265,0.094021,10,1,683.136354,""head of caudate nucleus""}","""ENCSR297HTV_ENCFF379TOS""","""ENCFF003BKC"""
"""chr6""",109705112,109705612,"""genic""","""AK9""",false,14150.0,0.280447,"{109691212,""ENSG00000155085"",0.708507,0.047177,1,2,0.0,""head of caudate nucleus""}","""ENCSR297HTV_ENCFF379TOS""","""ENCFF003BKC"""
"""chr3""",187743525,187744025,"""genic""","""BCL6""",false,1950.0,0.849752,"{187745725,""ENSG00000113916"",0.627262,0.131632,2,1,3918.463701,""head of caudate nucleus""}","""ENCSR297HTV_ENCFF379TOS""","""ENCFF003BKC"""
"""chr4""",7240273,7240773,"""genic""","""PSAPL1""",false,194450.0,0.355595,"{7434973,""ENSG00000178597"",0.329624,0.007315,56,1,373.480109,""head of caudate nucleus""}","""ENCSR297HTV_ENCFF379TOS""","""ENCFF003BKC"""
"""chr14""",21068386,21069251,"""intergenic""","""ARHGEF40""",false,1440.5,0.84484,"{21070259,""ENSG00000165801"",0.610447,0.16721,1,1,1552.684644,""head of caudate nucleus""}","""ENCSR297HTV_ENCFF379TOS""","""ENCFF003BKC"""
"""chr6""",43489283,43489783,"""genic""","""TJAP1""",false,12011.0,0.766338,"{43477522,""ENSG00000137221"",0.685788,0.088251,1,1,963.06376,""head of caudate nucleus""}","""ENCSR297HTV_ENCFF379TOS""","""ENCFF003BKC"""
"""chr9""",37027497,37027997,"""genic""","""PAX5""",false,6732.0,0.988384,"{37034479,""ENSG00000196092"",0.33889,0.186502,1,1,0.0,""head of caudate nucleus""}","""ENCSR297HTV_ENCFF379TOS""","""ENCFF003BKC"""
…,…,…,…,…,…,…,…,…,…,…
"""chr11""",118927938,118929923,"""intergenic""","""TTC36""",false,401498.5,0.338398,"{118527432,""ENSG00000172425"",0.341745,0.004349,40,10,2861.301186,""HCT116""}","""ENCSR547VBD_ENCFF547RWS""","""ENCFF999XQJ"""


In [15]:
#check
check_gene = encode_re2g_predictions.filter(pl.col("TargetGene") == "NEAT1")

print(f"datasets: {check_gene.select('re2g_dataset').n_unique()}")
display(check_gene)

datasets: 1459


re2g_chr,re2g_start,re2g_end,class,TargetGene,isSelfPromoter,distanceToTSS,ENCODE-rE2G.Score,re2g_features,re2g_dataset,bed_accession
cat,i64,i64,cat,str,bool,f64,f64,struct[8],cat,cat
"""chr11""",65487509,65488484,"""intergenic""","""NEAT1""",false,65199.5,0.569081,"{65422797,""ENSG00000245532"",0.837729,0.042705,19,1,2700.525672,""head of caudate nucleus""}","""ENCSR297HTV_ENCFF379TOS""","""ENCFF003BKC"""
"""chr11""",65421134,65421634,"""intergenic""","""NEAT1""",false,1413.0,0.753252,"{65422797,""ENSG00000245532"",0.837729,0.236116,1,1,15524.223624,""head of caudate nucleus""}","""ENCSR297HTV_ENCFF379TOS""","""ENCFF003BKC"""
"""chr11""",65419841,65420341,"""intergenic""","""NEAT1""",false,2706.0,0.775041,"{65422797,""ENSG00000245532"",0.837729,0.236116,2,1,15346.788678,""head of caudate nucleus""}","""ENCSR297HTV_ENCFF379TOS""","""ENCFF003BKC"""
"""chr11""",65418634,65419439,"""intergenic""","""NEAT1""",false,3760.5,0.888868,"{65422797,""ENSG00000245532"",0.837729,0.187372,3,1,13601.144934,""head of caudate nucleus""}","""ENCSR297HTV_ENCFF379TOS""","""ENCFF003BKC"""
"""chr11""",65477344,65477844,"""intergenic""","""NEAT1""",false,54797.0,0.476616,"{65422797,""ENSG00000245532"",0.837729,0.058649,15,1,3598.014654,""head of caudate nucleus""}","""ENCSR297HTV_ENCFF379TOS""","""ENCFF003BKC"""
"""chr11""",65416279,65416779,"""intergenic""","""NEAT1""",false,6268.0,0.825142,"{65422797,""ENSG00000245532"",0.837729,0.178763,4,1,9634.485978,""head of caudate nucleus""}","""ENCSR297HTV_ENCFF379TOS""","""ENCFF003BKC"""
"""chr11""",65422047,65423047,"""promoter""","""NEAT1""",true,250.0,0.999979,"{65422797,""ENSG00000245532"",0.837729,0.236116,0,0,9671.74533,""head of caudate nucleus""}","""ENCSR297HTV_ENCFF379TOS""","""ENCFF003BKC"""
…,…,…,…,…,…,…,…,…,…,…
"""chr11""",65471322,65472503,"""intergenic""","""NEAT1""",false,49115.5,0.274066,"{65422797,""ENSG00000245532"",0.920643,0.054935,17,1,1560.023955,""HCT116""}","""ENCSR547VBD_ENCFF547RWS""","""ENCFF999XQJ"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 4. Join (1) (Information: "E")

- E-G relationships のうち、enhancer 側での統合を以下のセルで実行する
- `join`するデータセットが両方とも非常に多くの行を持つため、一旦 `.lazy()`で `lazyframe`に変更し、`.collect()` で再度読み出す

In [16]:
# Lazy join keeps peak memory within the machine limit (about 100 GB measured)
enhancer_fantabio_join = (
    enhancer_fanta_intersect.lazy().join(
        encode_re2g_predictions.lazy(),
        on=["re2g_chr", "re2g_start", "re2g_end", "re2g_dataset"],
        how="inner"
    ).select(
        "cre_id", "cre_name", "cre_chr", "fantabio_cre_class","enhancer_cre_overlap_bp", "enhancer_cre_overlap_frac_of_fantabio_cre","TargetGene", "class","isSelfPromoter", "distanceToTSS", "ENCODE-rE2G.Score", "re2g_dataset",
        "Model", 
        pl.struct("Sample", "Biosample term name", "Biosample type", "Biosample term id_table12").alias("sample_info"), # metadata から統合したデータをまとめる
        pl.struct("re2g_chr", "re2g_start", "re2g_end").alias("re2g_enhancer_locus"), # まとめる
        pl.struct("cre_start", "cre_end", "directionality").alias("fantabio_enhancer_cre"), # まとめる
        "re2g_features"  # already a struct from encode side
    )
).sort(
    ["cre_id", "cre_name", "TargetGene", "re2g_dataset"]
).collect()

display(enhancer_fantabio_join)

cre_id,cre_name,cre_chr,fantabio_cre_class,enhancer_cre_overlap_bp,enhancer_cre_overlap_frac_of_fantabio_cre,TargetGene,class,isSelfPromoter,distanceToTSS,ENCODE-rE2G.Score,re2g_dataset,Model,sample_info,re2g_enhancer_locus,fantabio_enhancer_cre,re2g_features
str,str,cat,str,i64,f64,str,cat,bool,f64,f64,cat,cat,struct[4],struct[3],struct[3],struct[8]
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""promoter""",true,379.0,0.999972,"""ENCSR017RNR_ENCFF342KKO""","""DNase-only""","{""osteoblast_ENCSR000ELJ"",""osteoblast"",""primary cell"",""CL:0000062""}","{""chr1"",960336,961594}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.774799,0.259602,0,0,3748.21285,""osteoblast""}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""promoter""",true,685.0,0.999981,"""ENCSR021EXA_ENCFF114UJD""","""DNase-only""","{""endodermal_cell_ENCSR166KPV"",""endodermal cell"",""in vitro differentiated cells"",""CL:0000223""}","{""chr1"",960855,961687}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.460225,0.259602,0,0,604.370065,""endodermal cell""}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",294,0.617647,"""KLHL17""","""promoter""",true,197.5,0.999992,"""ENCSR028TTK_ENCFF690XUP""","""DNase-only""","{""mesothelial_cell_of_epicardium_ENCSR658EHM"",""mesothelial cell of epicardium"",""in vitro differentiated cells"",""CL:0011019""}","{""chr1"",960324,961243}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.467894,0.259602,0,0,353.837205,""mesothelial cell of epicardium""}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""promoter""",true,362.0,0.999969,"""ENCSR029NMT_ENCFF389ZHR""","""DNase-only""","{""GM19238_ENCSR000EJJ"",""GM19238"",""cell line"",""EFO:0002788""}","{""chr1"",960336,961560}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.798026,0.259602,0,0,789.67176,""GM19238""}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",302,0.634454,"""KLHL17""","""promoter""",true,196.5,0.999992,"""ENCSR031GVW_ENCFF034YIA""","""DNase-only""","{""adipocyte_ENCSR405TXU"",""adipocyte"",""in vitro differentiated cells"",""CL:0000136""}","{""chr1"",960314,961251}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.45495,0.259602,0,0,225.53406,""adipocyte""}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",308,0.647059,"""KLHL17""","""promoter""",true,196.0,0.999988,"""ENCSR042SRX_ENCFF159BAE""","""DNase-only""","{""activated_naive_CD8-positive__alpha-beta_T_cell_ENCSR518YYX"",""activated naive CD8-positive, alpha-beta T cell"",""primary cell"",""NTR:0000502""}","{""chr1"",960307,961257}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.586108,0.259602,0,0,510.821004,""activated naive CD8-positive, alpha-beta T cell""}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",329,0.691176,"""KLHL17""","""genic""",false,760.0,0.824621,"""ENCSR045WCT_ENCFF292SIC""","""DNase-only""","{""GM18507_ENCSR000EJI"",""GM18507"",""cell line"",""EFO:0002787""}","{""chr1"",961096,961596}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.570309,0.259602,1,1,1112.943352,""GM18507""}"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_99999""","""FCHS_99999""","""chr12""","""ELA""",289,0.802778,"""TSPAN19""","""genic""",false,116653.0,0.602168,"""ENCSR503SHG_ENCFF698IHL""","""DNase-only""","{""right_renal_pelvis_ENCSR986HEN"",""right renal pelvis"",""tissue"",""UBERON:0018116""}","{""chr12"",85152135,85153725}","{85153436,85153796,1.0}","{85036277,""ENSG00000231738"",0.676788,0.007547,9,2,20.567,""right renal pelvis""}"


In [17]:
print(f"ENCODE-rE2G overlap 1st quartile: {enhancer_fantabio_join.select("enhancer_cre_overlap_frac_of_fantabio_cre").to_series().quantile(0.25)}")
print(f"ENCODE-rE2G overlap median: {enhancer_fantabio_join.select("enhancer_cre_overlap_frac_of_fantabio_cre").to_series().median()}")
print(f"ENCODE-rE2G overlap 3rd quartile: {enhancer_fantabio_join.select("enhancer_cre_overlap_frac_of_fantabio_cre").to_series().quantile(0.75)}")

print(f"ENCODE-rE2G overlap mean: {enhancer_fantabio_join.select("ENCODE-rE2G.Score").to_series().mean()}")
print(f"ENCODE-rE2G overlap standard deviation: {enhancer_fantabio_join.select("ENCODE-rE2G.Score").to_series().std()}")

ENCODE-rE2G overlap 1st quartile: 0.8278145695364238
ENCODE-rE2G overlap median: 1.0
ENCODE-rE2G overlap 3rd quartile: 1.0
ENCODE-rE2G overlap mean: 0.6772379356752751
ENCODE-rE2G overlap standard deviation: 0.3108337504232726


&nbsp;

&nbsp;

### ENCODE rE2G score

- 中央値、第3四分位数などを調べておき、フィルタリングの条件に使用する
- ただし、それぞれのデータセットでは、異なる機械学習モデルを使用しており、あくまで参考程度に留めるべき値だと考えられる

In [18]:
print(f"ENCODE-rE2G score 1st quartile: {enhancer_fantabio_join['ENCODE-rE2G.Score'].quantile(0.25)}")
print(f"ENCODE-rE2G score median: {enhancer_fantabio_join['ENCODE-rE2G.Score'].median()}")
print(f"ENCODE-rE2G score 3rd quartile: {enhancer_fantabio_join['ENCODE-rE2G.Score'].quantile(0.75)}")
print(f"ENCODE-rE2G score mean: {enhancer_fantabio_join['ENCODE-rE2G.Score'].mean()}")
print(f"ENCODE-rE2G score standard deviation: {enhancer_fantabio_join['ENCODE-rE2G.Score'].std()}")

ENCODE-rE2G score 1st quartile: 0.3498146889984045
ENCODE-rE2G score median: 0.7343112189926492
ENCODE-rE2G score 3rd quartile: 0.9999702059721588
ENCODE-rE2G score mean: 0.6772379356752751
ENCODE-rE2G score standard deviation: 0.3108337504232726


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 5. Filter (1) (Information: "E") (IMPORTANT!)

- `ENCODE-rE2G`で予測された enhancer 領域と、`fanta.bio`で定義されているCREのカバー率に基づいてフィルタリング
- fanta.bio の CRE とのカバー率でフィルタリング
- `enhancer_cre_overlap_bp` / (`cre_end` - `cre_start`)

In [19]:
# The overlap filter is already applied in section 2, so this only frees the unfiltered join
enhancer_fantabio_join_filter = enhancer_fantabio_join
del enhancer_fantabio_join

display(enhancer_fantabio_join_filter)

cre_id,cre_name,cre_chr,fantabio_cre_class,enhancer_cre_overlap_bp,enhancer_cre_overlap_frac_of_fantabio_cre,TargetGene,class,isSelfPromoter,distanceToTSS,ENCODE-rE2G.Score,re2g_dataset,Model,sample_info,re2g_enhancer_locus,fantabio_enhancer_cre,re2g_features
str,str,cat,str,i64,f64,str,cat,bool,f64,f64,cat,cat,struct[4],struct[3],struct[3],struct[8]
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""promoter""",true,379.0,0.999972,"""ENCSR017RNR_ENCFF342KKO""","""DNase-only""","{""osteoblast_ENCSR000ELJ"",""osteoblast"",""primary cell"",""CL:0000062""}","{""chr1"",960336,961594}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.774799,0.259602,0,0,3748.21285,""osteoblast""}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""promoter""",true,685.0,0.999981,"""ENCSR021EXA_ENCFF114UJD""","""DNase-only""","{""endodermal_cell_ENCSR166KPV"",""endodermal cell"",""in vitro differentiated cells"",""CL:0000223""}","{""chr1"",960855,961687}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.460225,0.259602,0,0,604.370065,""endodermal cell""}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",294,0.617647,"""KLHL17""","""promoter""",true,197.5,0.999992,"""ENCSR028TTK_ENCFF690XUP""","""DNase-only""","{""mesothelial_cell_of_epicardium_ENCSR658EHM"",""mesothelial cell of epicardium"",""in vitro differentiated cells"",""CL:0011019""}","{""chr1"",960324,961243}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.467894,0.259602,0,0,353.837205,""mesothelial cell of epicardium""}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""promoter""",true,362.0,0.999969,"""ENCSR029NMT_ENCFF389ZHR""","""DNase-only""","{""GM19238_ENCSR000EJJ"",""GM19238"",""cell line"",""EFO:0002788""}","{""chr1"",960336,961560}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.798026,0.259602,0,0,789.67176,""GM19238""}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",302,0.634454,"""KLHL17""","""promoter""",true,196.5,0.999992,"""ENCSR031GVW_ENCFF034YIA""","""DNase-only""","{""adipocyte_ENCSR405TXU"",""adipocyte"",""in vitro differentiated cells"",""CL:0000136""}","{""chr1"",960314,961251}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.45495,0.259602,0,0,225.53406,""adipocyte""}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",308,0.647059,"""KLHL17""","""promoter""",true,196.0,0.999988,"""ENCSR042SRX_ENCFF159BAE""","""DNase-only""","{""activated_naive_CD8-positive__alpha-beta_T_cell_ENCSR518YYX"",""activated naive CD8-positive, alpha-beta T cell"",""primary cell"",""NTR:0000502""}","{""chr1"",960307,961257}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.586108,0.259602,0,0,510.821004,""activated naive CD8-positive, alpha-beta T cell""}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",329,0.691176,"""KLHL17""","""genic""",false,760.0,0.824621,"""ENCSR045WCT_ENCFF292SIC""","""DNase-only""","{""GM18507_ENCSR000EJI"",""GM18507"",""cell line"",""EFO:0002787""}","{""chr1"",961096,961596}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.570309,0.259602,1,1,1112.943352,""GM18507""}"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_99999""","""FCHS_99999""","""chr12""","""ELA""",289,0.802778,"""TSPAN19""","""genic""",false,116653.0,0.602168,"""ENCSR503SHG_ENCFF698IHL""","""DNase-only""","{""right_renal_pelvis_ENCSR986HEN"",""right renal pelvis"",""tissue"",""UBERON:0018116""}","{""chr12"",85152135,85153725}","{85153436,85153796,1.0}","{85036277,""ENSG00000231738"",0.676788,0.007547,9,2,20.567,""right renal pelvis""}"


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 6. Join (2) (Information: "G")

- target 側のデータ (fanta.bio との統合をおこなった状態)を統合する

&nbsp;

## 6-1: Fomatting for `join`

In [20]:
encodere2g_tatget_refseq_best_cre_select = encodere2g_tatget_refseq_best_cre.select(
    "TargetGene",
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    pl.struct("tss_chr", "tss_start", "tss_end", "tss_score", "tss_strand").alias("target_tss_info"),
    pl.struct("cre_chr", "cre_start", "cre_end", "cre_score", "cre_strand", "cre_thick_start", "cre_thick_end", "cre_rgb", "pooled_cre_scores", "directionality").alias("target_cre_info"),
    pl.struct("tss_cre_overlap_bp", "tss_cre_overlap_frac_of_tss").alias("target_cre_overlap_info")
).rename(
    {
        "cre_id": "target_cre_id",
        "cre_name": "target_cre_name",
        "fantabio_cre_class": "target_fantabio_cre_class"      
    }
)

display(encodere2g_tatget_refseq_best_cre_select)

TargetGene,target_cre_id,target_cre_name,target_fantabio_cre_class,target_tss_info,target_cre_info,target_cre_overlap_info
str,str,str,str,struct[5],struct[10],struct[2]
"""A1BG""","""FCHS_214135""","""cp1@A1BG""","""PLA""","{""chr19"",58353249,58353749,0,""-""}","{""chr19"",58353431,58353819,0,""."",58353468,58353803,""0,0,204"",""directionality:-0.999634|class:PLA"",-0.999634}","{318,0.636}"
"""A1CF""","""FCHS_49959""","""cp1@A1CF""","""PLA""","{""chr10"",50885425,50885925,0,""-""}","{""chr10"",50885528,50885774,0,""."",50885634,50885770,""0,0,204"",""directionality:-0.999645|class:PLA"",-0.999645}","{246,0.492}"
"""A2M""","""FCHS_88190""","""cp1@A2M""","""PLA""","{""chr12"",9115979,9116479,0,""-""}","{""chr12"",9115751,9116452,0,""."",9115913,9116283,""0,0,204"",""directionality:-0.999764|class:PLA"",-0.999764}","{473,0.946}"
"""A2M-AS1""","""FCHS_88157""","""cp2@A2M-AS1,cp1@LINC00612""","""PLA""","{""chr12"",9064926,9065426,0,""+""}","{""chr12"",9064873,9065385,0,""."",9065079,9065268,""0,0,102"",""directionality:-0.480891|class:PLA"",-0.480891}","{459,0.918}"
"""A2ML1""","""FCHS_88093""","""cp7@A2ML1""","""PLA""","{""chr12"",8822221,8822721,0,""+""}","{""chr12"",8822444,8822725,0,""."",8822444,8822620,""255,0,0"",""directionality:1|class:PLA"",1.0}","{277,0.554}"
"""A4GALT""","""FCHS_276044""","""cp4@A4GALT,cp5@CYB5R3""","""PLA""","{""chr22"",42720660,42721160,0,""-""}","{""chr22"",42720635,42721121,0,""."",42720828,42720942,""0,0,204"",""directionality:-0.999079|class:PLA"",-0.999079}","{461,0.922}"
"""AAAS""","""FCHS_95187""","""cp11@AAAS""","""PLA""","{""chr12"",53321378,53321878,0,""-""}","{""chr12"",53321460,53322000,0,""."",53321591,53321702,""0,0,204"",""directionality:-0.989867|class:PLA"",-0.989867}","{418,0.836}"
…,…,…,…,…,…,…
"""ZXDB""","""FCHS_440127""","""cp3@ZXDB""","""ELA""","{""chrX"",57591585,57592085,0,""+""}","{""chrX"",57591492,57591853,0,""."",57591614,57591733,""255,202,74"",""directionality:0.939394|class:ELA"",0.939394}","{268,0.536}"


&nbsp;

&nbsp;

## 6-2: Join

In [21]:
e2g_join = enhancer_fantabio_join_filter.join(
    encodere2g_tatget_refseq_best_cre_select,
    on=["TargetGene"],
    how="inner"
).select(
    # 見やすいように列順を変更
    "ENCODE-rE2G.Score",
    "re2g_dataset",
    "Model",
    "sample_info",
    "cre_id",
    "cre_name",
    "cre_chr",
    "fantabio_cre_class",
    "enhancer_cre_overlap_bp",
    "enhancer_cre_overlap_frac_of_fantabio_cre",
    "TargetGene",
    "target_cre_id",
    "target_cre_name",
    "target_fantabio_cre_class",
    "class",
    "isSelfPromoter",
    "distanceToTSS",
    "re2g_enhancer_locus",
    "fantabio_enhancer_cre",
    # "re2g_features",
    "target_tss_info",
    "target_cre_info",
    "target_cre_overlap_info"
)

display(e2g_join)

ENCODE-rE2G.Score,re2g_dataset,Model,sample_info,cre_id,cre_name,cre_chr,fantabio_cre_class,enhancer_cre_overlap_bp,enhancer_cre_overlap_frac_of_fantabio_cre,TargetGene,target_cre_id,target_cre_name,target_fantabio_cre_class,class,isSelfPromoter,distanceToTSS,re2g_enhancer_locus,fantabio_enhancer_cre,target_tss_info,target_cre_info,target_cre_overlap_info
f64,cat,cat,struct[4],str,str,cat,str,i64,f64,str,str,str,str,cat,bool,f64,struct[3],struct[3],struct[5],struct[10],struct[2]
0.999972,"""ENCSR017RNR_ENCFF342KKO""","""DNase-only""","{""osteoblast_ENCSR000ELJ"",""osteoblast"",""primary cell"",""CL:0000062""}","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""promoter""",true,379.0,"{""chr1"",960336,961594}","{960949,961425,-0.483871}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.999981,"""ENCSR021EXA_ENCFF114UJD""","""DNase-only""","{""endodermal_cell_ENCSR166KPV"",""endodermal cell"",""in vitro differentiated cells"",""CL:0000223""}","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""promoter""",true,685.0,"{""chr1"",960855,961687}","{960949,961425,-0.483871}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.999992,"""ENCSR028TTK_ENCFF690XUP""","""DNase-only""","{""mesothelial_cell_of_epicardium_ENCSR658EHM"",""mesothelial cell of epicardium"",""in vitro differentiated cells"",""CL:0011019""}","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",294,0.617647,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""promoter""",true,197.5,"{""chr1"",960324,961243}","{960949,961425,-0.483871}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.999969,"""ENCSR029NMT_ENCFF389ZHR""","""DNase-only""","{""GM19238_ENCSR000EJJ"",""GM19238"",""cell line"",""EFO:0002788""}","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""promoter""",true,362.0,"{""chr1"",960336,961560}","{960949,961425,-0.483871}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.999992,"""ENCSR031GVW_ENCFF034YIA""","""DNase-only""","{""adipocyte_ENCSR405TXU"",""adipocyte"",""in vitro differentiated cells"",""CL:0000136""}","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",302,0.634454,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""promoter""",true,196.5,"{""chr1"",960314,961251}","{960949,961425,-0.483871}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.999988,"""ENCSR042SRX_ENCFF159BAE""","""DNase-only""","{""activated_naive_CD8-positive__alpha-beta_T_cell_ENCSR518YYX"",""activated naive CD8-positive, alpha-beta T cell"",""primary cell"",""NTR:0000502""}","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",308,0.647059,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""promoter""",true,196.0,"{""chr1"",960307,961257}","{960949,961425,-0.483871}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.824621,"""ENCSR045WCT_ENCFF292SIC""","""DNase-only""","{""GM18507_ENCSR000EJI"",""GM18507"",""cell line"",""EFO:0002787""}","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",329,0.691176,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""genic""",false,760.0,"{""chr1"",961096,961596}","{960949,961425,-0.483871}","{""chr1"",960336,960836

In [22]:
e2g_join.write_parquet(
    f"../../out/ENCODE_rE2G/parquet_file/encode_re2g_ctrl_encode_predictions_v{execute_date}.parquet",
    compression="zstd",
    compression_level=9
)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 7.  Enhancer-target Region Unique Count

- enhancer 側で統合した `fanta.bio` の `cre_id` と Target 側の領域として統合した `fanta.bio`の `target_cre_id` の unique pair をカウントする
- `re2g_dataset` カラムのカウントも行う

In [25]:
e2g_join_unique_count = e2g_join.group_by(
    "cre_id",
    "fantabio_cre_class",
    "target_cre_id",
    "target_cre_name",
    "target_fantabio_cre_class",
    # "isSelfPromoter" (この値は遺伝子ごとに異なるのでグループ化しない)
).agg(
    pl.col("re2g_dataset").unique().alias("re2g_dataset"),
    pl.col("re2g_dataset").n_unique().alias("re2g_dataset_count"),
    pl.col("ENCODE-rE2G.Score").alias("ENCODE-rE2G.Score"),
    
    pl.col("ENCODE-rE2G.Score").min().alias("ENCODE-rE2G.Score_min"),
    pl.col("ENCODE-rE2G.Score").max().alias("ENCODE-rE2G.Score_max"),
    pl.col("ENCODE-rE2G.Score").mean().alias("ENCODE-rE2G.Score_mean"),

    # pl.col("ENCODE-rE2G.Score").quantile(0.25).alias("ENCODE-rE2G.Score_q1"),
    # pl.col("ENCODE-rE2G.Score").median().alias("ENCODE-rE2G.Score_median"),
    # pl.col("ENCODE-rE2G.Score").quantile(0.75).alias("ENCODE-rE2G.Score_q3"),
    # pl.col("ENCODE-rE2G.Score").std().alias("ENCODE-rE2G.Score_std"),
    
).sort(
    by=["re2g_dataset_count"],
    descending=True
)


print(f"ENCODE-rE2G enhancer - target region interaction unique pair count: {e2g_join_unique_count.select(['cre_id', 'target_cre_id']).n_unique()}")
display(e2g_join_unique_count)

ENCODE-rE2G enhancer - target region interaction unique pair count: 1568858


cre_id,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean
str,str,str,str,str,list[cat],u32,list[f64],f64,f64,f64
"""FCHS_341992""","""PLA""","""FCHS_341992""","""cp1@TIGD6""","""PLA""","[""ENCSR000MZA_ENCFF654AMU"", ""ENCSR002YWN_ENCFF435DPV"", … ""ENCSR999YQC_ENCFF625FFH""]",1460,"[0.999997, 1.0, … 0.999984]",0.936416,1.0,0.999943
"""FCHS_97067""","""PLA""","""FCHS_97067""","""cp3@ATP23,cp2@GIHCG""","""PLA""","[""ENCSR000MZA_ENCFF654AMU"", ""ENCSR002YWN_ENCFF435DPV"", … ""ENCSR999YQC_ENCFF625FFH""]",1460,"[0.999984, 0.999965, … 0.999998]",0.966933,1.0,0.999959
"""FCHS_351197""","""PLA""","""FCHS_351197""","""cp1@NUP153,cp1@NUP153-AS1""","""PLA""","[""ENCSR000MZA_ENCFF654AMU"", ""ENCSR002YWN_ENCFF435DPV"", … ""ENCSR999YQC_ENCFF625FFH""]",1460,"[0.999967, 0.999861, … 0.999876]",0.964552,0.999999,0.999869
"""FCHS_418453""","""PLA""","""FCHS_418453""","""cp5@SMIM27,cp2@TOPORS""","""PLA""","[""ENCSR000MZA_ENCFF654AMU"", ""ENCSR002YWN_ENCFF435DPV"", … ""ENCSR999YQC_ENCFF625FFH""]",1460,"[0.999982, 0.99999, … 0.999955]",0.992224,0.999996,0.99997
"""FCHS_304241""","""PLA""","""FCHS_304241""","""cp2@TFRC""","""PLA""","[""ENCSR000MZA_ENCFF654AMU"", ""ENCSR002YWN_ENCFF435DPV"", … ""ENCSR999YQC_ENCFF625FFH""]",1460,"[0.999997, 0.999998, … 0.999999]",0.953091,1.0,0.999955
"""FCHS_204737""","""PLA""","""FCHS_204737""","""cp1@ANKRD27,cp2@RGS9BP""","""PLA""","[""ENCSR000MZA_ENCFF654AMU"", ""ENCSR002YWN_ENCFF435DPV"", … ""ENCSR999YQC_ENCFF625FFH""]",1460,"[0.999779, 0.999873, … 0.999999]",0.970858,1.0,0.999907
"""FCHS_26041""","""PLA""","""FCHS_26041""","""cp4@TRIM46""","""PLA""","[""ENCSR000MZA_ENCFF654AMU"", ""ENCSR002YWN_ENCFF435DPV"", … ""ENCSR999YQC_ENCFF625FFH""]",1460,"[0.999986, 0.999992, … 0.999991]",0.966805,1.0,0.999962
…,…,…,…,…,…,…,…,…,…,…
"""FCHS_583025""","""PLA""","""FCHS_413495""","""cp1@KHDRBS3""","""PLA""","[""ENCSR132MYN_ENCFF758HBA""]",1,[0.236961],0.236961,0.236961,0.236961


In [26]:
# check (ENEMAL)
FCHS_73649 = e2g_join_unique_count.filter(pl.col("cre_id") == "FCHS_73649")
display(FCHS_73649)

cre_id,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean
str,str,str,str,str,list[cat],u32,list[f64],f64,f64,f64
"""FCHS_73649""","""PLA""","""FCHS_73694""","""cp7@MALAT1""","""PLA""","[""ENCSR000MZA_ENCFF654AMU"", ""ENCSR002YWN_ENCFF435DPV"", … ""ENCSR999YQC_ENCFF625FFH""]",1324,"[0.283167, 0.639376, … 0.219676]",0.202682,0.964458,0.539182
"""FCHS_73649""","""PLA""","""FCHS_470991""","""cp8@TALAM1""","""PLA""","[""ENCSR002YWN_ENCFF435DPV"", ""ENCSR004ZDG_ENCFF053LFW"", … ""ENCSR998FKR_ENCFF868ANT""]",1160,"[0.681613, 0.844346, … 0.845648]",0.201135,0.92242,0.513754
"""FCHS_73649""","""PLA""","""FCHS_73545""","""cp2@NEAT1""","""PLA""","[""ENCSR002YWN_ENCFF435DPV"", ""ENCSR004ZDG_ENCFF053LFW"", … ""ENCSR998GYO_ENCFF308ZWK""]",1150,"[0.396699, 0.595376, … 0.39528]",0.201427,0.883448,0.390309
"""FCHS_73649""","""PLA""","""FCHS_73518""","""cp2@FRMD8""","""PLA""","[""ENCSR002YWN_ENCFF435DPV"", ""ENCSR004ZDG_ENCFF053LFW"", … ""ENCSR998GYO_ENCFF308ZWK""]",1051,"[0.247026, 0.387245, … 0.412991]",0.201081,0.592428,0.321606
"""FCHS_73649""","""PLA""","""FCHS_73510""","""cp12@SLC25A45""","""PLA""","[""ENCSR002YWN_ENCFF435DPV"", ""ENCSR004ZDG_ENCFF053LFW"", … ""ENCSR998GYO_ENCFF308ZWK""]",509,"[0.206821, 0.31069, … 0.336574]",0.201748,0.629218,0.28773
"""FCHS_73649""","""PLA""","""FCHS_73767""","""cp18@LTBP3""","""PLA""","[""ENCSR025QUQ_ENCFF917GDE"", ""ENCSR118NOT_ENCFF078CFD"", … ""ENCSR961MWH_ENCFF177KBG""]",20,"[0.352282, 0.251112, … 0.248446]",0.201435,0.352282,0.232302
"""FCHS_73649""","""PLA""","""FCHS_73490""","""cp1@TIGD3""","""PLA""","[""ENCSR153XYE_ENCFF663ISA"", ""ENCSR357UZR_ENCFF343OAT"", … ""ENCSR961MWH_ENCFF177KBG""]",6,"[0.280829, 0.217786, … 0.225584]",0.215491,0.448005,0.272415
…,…,…,…,…,…,…,…,…,…,…
"""FCHS_73649""","""PLA""","""FCHS_71523""","""cp4@CD6""","""PLA""","[""ENCSR997TVR_ENCFF547ARY""]",1,[0.461343],0.461343,0.461343,0.461343


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 8. Check `isSelfPromoter`

&nbsp;

&nbsp;

&nbsp;

## 8-0: Selection of proximal enhancer

- `isSelfPromoter == True` でフィルタリングする

In [ ]:
# e2g_join_self_promoter = e2g_join.filter(
#     pl.col("isSelfPromoter") == True
# )

# display(e2g_join_self_promoter)

&nbsp;

&nbsp;

&nbsp;

## 8-1: Selection of distal enhancer (Important!)

- `TargetGene` の近傍ではない enhancer を標的とする
- `isSelfPromoter` == False でフィルタリング
- kernel がクラッシュするので、ここでは実行しない

In [ ]:
# e2g_join_filter = e2g_join.filter(
#     pl.col("isSelfPromoter") == False
# )

# display(e2g_join_filter)

: 

In [ ]:
# e2g_join_filter_unique_count = e2g_join_filter.group_by(
#     "cre_id",
#     "cre_name",
#     "fantabio_cre_class",
#     "target_cre_id",
#     "target_cre_name",
#     "target_fantabio_cre_class"
# ).agg(
#     pl.col("re2g_dataset").unique().alias("re2g_dataset"),
#     pl.col("re2g_dataset").n_unique().alias("re2g_dataset_count"),
#     pl.col("ENCODE-rE2G.Score").alias("ENCODE-rE2G.Score"),
    
#     pl.col("ENCODE-rE2G.Score").min().alias("ENCODE-rE2G.Score_min"),
#     pl.col("ENCODE-rE2G.Score").max().alias("ENCODE-rE2G.Score_max"),
#     pl.col("ENCODE-rE2G.Score").mean().alias("ENCODE-rE2G.Score_mean"),

#     # pl.col("ENCODE-rE2G.Score").quantile(0.25).alias("ENCODE-rE2G.Score_q1"),
#     # pl.col("ENCODE-rE2G.Score").median().alias("ENCODE-rE2G.Score_median"),
#     # pl.col("ENCODE-rE2G.Score").quantile(0.75).alias("ENCODE-rE2G.Score_q3"),
#     # pl.col("ENCODE-rE2G.Score").std().alias("ENCODE-rE2G.Score_std"),
# ).sort(
#     by=["re2g_dataset_count"],
#     descending=True
# )

# pl.Config.set_tbl_rows(10)
# print(f"ENCODE-rE2G enhancer - target region interaction unique pair count: {e2g_join_filter_unique_count.select(['cre_id', 'target_cre_id']).n_unique()}")
# display(e2g_join_filter_unique_count)